# x27 Efficient Data Loading Demo

This notebook demonstrates how to use the new memory-efficient data loading approach.

## Problem with the Old Approach

The old approach in `generate_data.py` duplicates image features across thousands of samples:
- ~60 images × 100×100×3 pixels × 4 bytes = ~7 MB of unique image data
- But with 1000 samples × 2 graphs × 15 nodes = 30,000 node features
- So we stored ~30,000 × (100×100×3×4) bytes = ~3.6 GB of duplicated data!

## New Efficient Approach

The new approach stores:
1. **Image features cache** (~7 MB): Image features computed once, stored once
2. **Lightweight samples** (~few MB): Just store image IDs and transformations

Graph data is constructed on-the-fly during training by looking up image features from the cache.


In [32]:
%load_ext autoreload
%autoreload 2

import pathlib
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cuda


## Option 1: Generate New Efficient Data

If you haven't generated the efficient data yet, run `generate_data_efficient.py` first:

```bash
uv run python generate_data_efficient.py
```

Or generate it directly in this notebook:


In [33]:
# Only run this cell if you need to generate new data
# Otherwise, skip to Option 2: Load Existing Data

from generate_data_efficient import generate_sfm_data_efficient

model_path = pathlib.Path('./output/01_nerf/02_lego_large/1762043798762/0')
image_dir = pathlib.Path('data/01_nerf/02_lego_large')
output_dir = pathlib.Path('data/efficient')

# Generate efficient data
dataset = generate_sfm_data_efficient(
    model_path=model_path,
    image_dir=image_dir,
    num_samples=1000,
    subset_size=15,
    include_image_features=True,
    img_size=100,
    translation_range=(-1.0, 1.0),
    random_seed=42,
    output_dir=output_dir,
)


Loaded reconstruction with 69 images

Precomputing image features at 100x100...
Saved 69 image features to data\efficient\image_features.pt
Image features size: 7.90 MB (stored once)

Generating 1000 samples...


Generating samples:   1%|          | 12/1000 [00:06<08:47,  1.87it/s]


KeyboardInterrupt: 

## Option 2: Load Existing Efficient Data


In [34]:
from generate_data_efficient import load_efficient_dataset

output_dir = pathlib.Path('data/efficient')

# Load the efficient dataset
dataset = load_efficient_dataset(output_dir, include_image_features=True)
print(f"Loaded dataset with {len(dataset)} samples")
print(f"Metadata: {dataset.metadata}")


Loaded dataset with 1000 samples
Metadata: {'num_samples': 1000, 'subset_size': 15, 'include_image_features': True, 'img_size': 100, 'translation_range': (-10.0, 10.0), 'random_seed': 42, 'model_path': 'output\\01_nerf\\02_lego_large\\1762043798762\\0', 'image_dir': 'data\\01_nerf\\02_lego_large'}


## Inspect Sample Data

Let's look at what a single sample looks like:


In [35]:
# Get a sample
graph_a, graph_b, label_trans, label_quat = dataset[0]

print(f"Graph A:")
print(f"  - Node features shape: {graph_a.x.shape}")
print(f"  - Edge index shape: {graph_a.edge_index.shape}")
print(f"  - Number of nodes: {graph_a.x.shape[0]}")
print(f"  - Number of edges: {graph_a.edge_index.shape[1]}")

print(f"\nGraph B:")
print(f"  - Node features shape: {graph_b.x.shape}")
print(f"  - Edge index shape: {graph_b.edge_index.shape}")

print(f"\nLabels:")
print(f"  - Translation: {label_trans}")
print(f"  - Quaternion (xyzw): {label_quat}")


Graph A:
  - Node features shape: torch.Size([15, 30007])
  - Edge index shape: torch.Size([2, 166])
  - Number of nodes: 15
  - Number of edges: 166

Graph B:
  - Node features shape: torch.Size([15, 30007])
  - Edge index shape: torch.Size([2, 202])

Labels:
  - Translation: tensor([ 0.4902, -0.9442,  7.6850])
  - Quaternion (xyzw): tensor([ 0.2865, -0.0798,  0.3736,  0.8786])


## Visualize Sample Data

In [37]:
import utils.vis
from utils.loaders.efficient import sfm_sample_to_data_sfm

# Convert to our "legacy" data format for visualization
sample = dataset.samples[4]

graph_a_sfm, graph_b_sfm = sfm_sample_to_data_sfm(sample)
utils.vis.plot_3D_graph_sfm(graph_a_sfm, graph_b_sfm)

## Use with DataLoader for Training

The efficient dataset works seamlessly with PyG's DataLoader:


In [38]:
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

# Split into train/val
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


Train batches: 100
Val batches: 25


In [39]:
# Test that batching works correctly
for batch in train_loader:
    graph_a, graph_b, label_trans, label_quat = batch
    print(f"Batch:")
    print(f"  Graph A: {graph_a}")
    print(f"  Graph B: {graph_b}")
    print(f"  Label translation shape: {label_trans.shape}")
    print(f"  Label quaternion shape: {label_quat.shape}")
    break  # Just show the first batch


Batch:
  Graph A: DataBatch(x=[120, 30007], edge_index=[2, 1540], batch=[120], ptr=[9])
  Graph B: DataBatch(x=[120, 30007], edge_index=[2, 1584], batch=[120], ptr=[9])
  Label translation shape: torch.Size([8, 3])
  Label quaternion shape: torch.Size([8, 4])


## Training Example

Here's a minimal training loop example using the efficient dataset:


In [41]:
from models.models import SiameseGAT_v7
from utils.loss import compute_combined_loss
from tqdm import tqdm
import roma

# Get feature dimension from first sample
sample_graph_a, _, _, _ = dataset[0]
in_features = sample_graph_a.x.shape[1]
print(f"Input feature dimension: {in_features}")

# Create model
model = SiameseGAT_v7(
    num_node_features=in_features,
    graph_embedding_dim=64
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Loss functions (same as x26)
criterion_translation = torch.nn.MSELoss()
criterion_rotation = torch.nn.L1Loss()

# Training loop (just a few epochs for demo)
model.train()
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        graph_a, graph_b, label_trans, label_quat = batch

        # Move to device
        graph_a = graph_a.to(device)
        graph_b = graph_b.to(device)
        label_trans = label_trans.to(device)
        label_quat = label_quat.to(device)

        optimizer.zero_grad()

        # Forward pass
        pred_trans, pred_rot_raw = model(graph_a, graph_b)

        # Normalize the predicted matrix to be a valid SO(3) rotation matrix
        pred_rot = roma.special_procrustes(pred_rot_raw)

        # Compute loss
        loss, loss_trans, loss_rot = compute_combined_loss(
            pred_trans, pred_rot,
            label_trans, label_quat,
            criterion_translation, criterion_rotation,
            lambda_translation=1.0
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Average Loss: {total_loss / len(train_loader):.4f}")


Input feature dimension: 30007


Epoch 1: 100%|██████████| 100/100 [00:04<00:00, 20.75it/s]


Epoch 1 - Average Loss: 34.3869


Epoch 2: 100%|██████████| 100/100 [00:04<00:00, 20.06it/s]


Epoch 2 - Average Loss: 34.2271


Epoch 3: 100%|██████████| 100/100 [00:04<00:00, 20.74it/s]

Epoch 3 - Average Loss: 34.2088
